# Chapitre 7 — GroupBy et tableaux croisés

## Le problème

Le chiffre d'affaires total est une information ; le chiffre d'affaires **par canal**, **par mois** ou **par client** est ce qui permet de décider. En SQL, c'est le rôle de `GROUP BY`. En Pandas, c'est `groupby`.

L'opération se décompose en trois temps, connus sous le nom *split-apply-combine* :

```
        commandes                    split                 apply            combine
 ┌─────────┬─────────┐      ┌─────────┬─────────┐
 │ canal   │ montant │      │ web     │   79.0  │
 ├─────────┼─────────┤      │ web     │  549.0  │ → sum → 628.0  ┐
 │ web     │   79.0  │      └─────────┴─────────┘                │     ┌──────────┬────────┐
 │ web     │  549.0  │  →   ┌─────────┬─────────┐                ├ →   │ magasin  │  54.4  │
 │ magasin │   29.9  │      │ magasin │   29.9  │ → sum →  54.4  ┘     │ web      │ 628.0  │
 │ magasin │   24.5  │      │ magasin │   24.5  │                      └──────────┴────────┘
 └─────────┴─────────┘      └─────────┴─────────┘
```

1. **split** : découper la table en groupes selon une ou plusieurs clés ;
2. **apply** : appliquer une agrégation (ou une transformation) à chaque groupe ;
3. **combine** : rassembler les résultats dans une nouvelle table, indexée par les clés.

In [ ]:
import numpy as np
import pandas as pd

commandes = pd.read_csv("../donnees/demo/commandes.csv", parse_dates=["date_commande"])
produits = pd.read_csv("../donnees/demo/produits.csv")
commandes["montant"] = commandes["quantite"] * commandes["produit_id"].map(produits.set_index("produit_id")["prix"])
jo = pd.read_csv("../donnees/bstorm/Exercices_Pandas_athlete_events.csv")

## 1. Un regroupement simple

In [ ]:
commandes.groupby("canal")["montant"].sum()

Le résultat est une Series dont l'**index est formé des valeurs de la clé**. Équivalent SQL :

```sql
SELECT canal, SUM(montant) FROM commandes GROUP BY canal;
```

L'objet intermédiaire `commandes.groupby("canal")` ne calcule rien tant qu'on ne lui demande pas d'agréger :

In [ ]:
groupes = commandes.groupby("canal")
groupes

In [ ]:
groupes.ngroups, groupes.size()

### `size` ou `count` ?

> **Prédiction.** On ajoute une valeur manquante dans la colonne `montant`. `groupby("canal").size()` et `groupby("canal")["montant"].count()` donneront-ils le même résultat ?

In [ ]:
essai = commandes.copy()
essai.loc[0, "montant"] = np.nan
pd.DataFrame({"size": essai.groupby("canal").size(), "count": essai.groupby("canal")["montant"].count()})

- `size()` compte les **lignes** de chaque groupe (`COUNT(*)`) ;
- `count()` compte les valeurs **non manquantes** d'une colonne (`COUNT(col)`).

## 2. Plusieurs agrégations, plusieurs clés

### Agrégations nommées

In [ ]:
par_canal = commandes.groupby("canal").agg(
    nb_commandes=("commande_id", "count"),
    ca=("montant", "sum"),
    panier_moyen=("montant", "mean"),
    nb_clients=("client_id", "nunique"),
)
par_canal.sort_values("ca", ascending=False).round(1)

### Garder la clé en colonne : `as_index=False`

In [ ]:
commandes.groupby("canal", as_index=False)["montant"].sum()

Le résultat est une table « plate », prête à être exportée ou jointe. `reset_index()` sur le résultat produit le même effet.

### Plusieurs clés : un index hiérarchique

In [ ]:
canal_statut = commandes.groupby(["canal", "statut"])["montant"].sum()
canal_statut

L'index comporte deux niveaux : c'est un **MultiIndex**. On sélectionne avec un tuple, ou on le remet à plat :

In [ ]:
canal_statut.loc[("web", "livrée")]

In [ ]:
canal_statut.reset_index().head()

Le chapitre 9 montrera `unstack`, qui déplace un niveau d'index en colonnes.

### Plusieurs fonctions par colonne : des colonnes à deux niveaux

In [ ]:
stats = commandes.groupby("canal").agg({"montant": ["sum", "mean"], "quantite": ["sum", "max"]})
stats

In [ ]:
stats.columns

Les colonnes forment aussi un MultiIndex : `("montant", "sum")`, `("montant", "mean")`… On y accède par tuple, ou on aplatit les noms :

In [ ]:
stats[("montant", "mean")]

In [ ]:
stats.columns = ["_".join(col) for col in stats.columns]
stats.reset_index()

Les agrégations nommées évitent ce problème dès le départ : on les préfère quand le résultat doit être exploité ensuite.

## 3. Regrouper par une date

On peut regrouper par une expression calculée, par exemple le mois de la commande :

In [ ]:
commandes.groupby(commandes["date_commande"].dt.month)["montant"].sum().head()

## 4. Filtrer après l'agrégation (`HAVING`)

En SQL, `WHERE` filtre les lignes avant le regroupement et `HAVING` filtre les groupes après. En Pandas, on filtre simplement le résultat :

```sql
SELECT client_id, SUM(montant) AS ca FROM commandes GROUP BY client_id HAVING SUM(montant) > 2000;
```

In [ ]:
ca_client = commandes.groupby("client_id")["montant"].sum()
ca_client[ca_client > 2000].sort_values(ascending=False)

## 5. `transform` : une valeur de groupe sur chaque ligne

`agg` renvoie **une ligne par groupe**. `transform` renvoie **un résultat par ligne d'origine**, en répétant la valeur du groupe. C'est la technique de base pour créer des variables pour un modèle : part d'une commande dans le total du client, écart à la moyenne du groupe…

In [ ]:
commandes["ca_client"] = commandes.groupby("client_id")["montant"].transform("sum")
commandes["part_du_client"] = (commandes["montant"] / commandes["ca_client"]).round(3)
commandes[["commande_id", "client_id", "montant", "ca_client", "part_du_client"]].head()

## 6. Parcourir les groupes

Pour comprendre ce que contient chaque groupe, on peut les parcourir. Comme pour `iterrows`, c'est un outil d'inspection, pas de calcul.

In [ ]:
for canal, groupe in commandes.groupby("canal"):
    print(canal, groupe.shape, groupe["montant"].sum().round(1))

In [ ]:
commandes.groupby("canal")["montant"].describe().round(1)

## 7. Regrouper sur une colonne `category`

Pour une colonne catégorielle, le paramètre `observed` indique s'il faut afficher les catégories déclarées mais absentes des données.

In [ ]:
canal_cat = commandes["canal"].astype(pd.CategoricalDtype(["web", "magasin", "telephone", "marketplace"]))
commandes.groupby(canal_cat, observed=False)["montant"].sum()

In [ ]:
commandes.groupby(canal_cat)["montant"].sum()           # observed=True par défaut en Pandas 3

En Pandas 1 et 2, le défaut était `observed=False` et un avertissement demandait de préciser le paramètre. Depuis Pandas 3, seules les catégories présentes apparaissent par défaut.

## 8. Granularité : que compte-t-on vraiment ?

Dans le fichier JO, une ligne est **une participation d'un athlète à une épreuve**. Une équipe de football médaillée d'or produit une ligne par joueur.

> **Pourquoi ?** Combien de médailles d'or l'Argentine a-t-elle gagnées en football en 2004 ? Le fichier contient combien de lignes pour cet événement ?

In [ ]:
foot_arg = jo[(jo["NOC"] == "ARG") & (jo["Sport"] == "Football") & (jo["Year"] == 2004) & (jo["Medal"] == "Gold")]
len(foot_arg), foot_arg[["Event", "Year", "Medal"]].drop_duplicates()

Compter les lignes revient à compter **les médaillés**, pas **les médailles**. Pour compter les médailles d'un pays, on garde une ligne par combinaison (épreuve, édition, équipe, médaille) :

In [ ]:
medailles = jo.dropna(subset=["Medal"])
medailles_uniques = medailles.drop_duplicates(subset=["Event", "Games", "NOC", "Medal"])
pd.DataFrame({
    "lignes (médaillés)": medailles.groupby("NOC").size(),
    "médailles": medailles_uniques.groupby("NOC").size(),
}).sort_values("médailles", ascending=False).head()

In [ ]:
belgique = pd.Series({"lignes": (medailles["NOC"] == "BEL").sum(),
                      "médailles": (medailles_uniques["NOC"] == "BEL").sum()})
belgique

Aucune des deux réponses n'est fausse : elles répondent à deux questions différentes. Avant tout `groupby`, il faut savoir ce qu'est une ligne et ce qu'on veut compter.

## 9. Le tableau croisé : `pivot_table`

Un tableau croisé place une clé en lignes, une autre en colonnes, et une agrégation dans les cellules.

In [ ]:
pd.pivot_table(commandes, index="canal", columns="statut", values="montant", aggfunc="sum").round(0)

| Paramètre | Rôle |
|---|---|
| `index` | clé(s) en lignes |
| `columns` | clé(s) en colonnes |
| `values` | colonne(s) à agréger |
| `aggfunc` | fonction (`"sum"`, `"mean"`, `"count"`, une liste…) |
| `fill_value` | valeur pour les combinaisons absentes |
| `margins=True` | ajoute une ligne et une colonne `All` (totaux) |

In [ ]:
medailles_uniques_pays = medailles_uniques[medailles_uniques["NOC"].isin(["USA", "URS", "GER", "FRA", "BEL"])]
pd.pivot_table(medailles_uniques_pays, index="NOC", columns="Medal", values="ID",
               aggfunc="count", fill_value=0, margins=True)

La ligne et la colonne `All` contiennent les totaux. Attention en exploitant le résultat : si on trie ou on cherche un maximum, la ligne `All` sera toujours en tête. On la retire avec `.drop(index="All")`.

`pd.crosstab` est un raccourci pour compter des combinaisons :

In [ ]:
pd.crosstab(commandes["canal"], commandes["statut"], normalize="index").round(2)

## 10. Un indicateur dérivé : médailles par athlète

Question 32 de l'exercice JO : quels pays étaient les plus « efficaces » de 1952 à 1988, c'est-à-dire avec le plus de médailles par athlète envoyé ? Il faut deux agrégations différentes, puis un ratio.

In [ ]:
periode = jo[jo["Year"].between(1952, 1988)]
efficacite = pd.DataFrame({
    "athletes": periode.groupby("NOC")["ID"].nunique(),
    "medailles": periode.dropna(subset=["Medal"])
                        .drop_duplicates(subset=["Event", "Games", "NOC", "Medal"])
                        .groupby("NOC").size(),
}).fillna(0)
efficacite["medailles_par_athlete"] = (efficacite["medailles"] / efficacite["athletes"]).round(3)
efficacite[efficacite["athletes"] >= 100].sort_values("medailles_par_athlete", ascending=False).head()

Les choix de définition (athlètes distincts ou participations, médailles ou médaillés, seuil minimal d'athlètes) changent le classement. Les documenter fait partie de la réponse.

## Pièges fréquents

| Situation | Symptôme | Réflexe |
|---|---|---|
| Compter des lignes au lieu d'entités | résultats gonflés (sports collectifs) | définir la granularité, `drop_duplicates(subset=...)` |
| `count()` au lieu de `size()` | groupes sous-comptés | `count` ignore les `NaN` |
| Colonnes MultiIndex après `agg` avec listes | accès difficile, export illisible | agrégations nommées, ou aplatir les noms |
| Ligne `All` d'un `pivot_table` avec `margins=True` | toujours en tête d'un tri | `.drop(index="All")` |
| Agréger une colonne texte | `TypeError` | sélectionner la colonne avant l'agrégation |
| `agg` vs `transform` | mauvais nombre de lignes | `transform` garde une ligne par ligne d'origine |

## Exercices

- Exercice Bstorm **JO**, questions **23 à 32**.
- Exercice Bstorm **Sakila**, questions **9 et 10**.
- Exercice supplémentaire **S7** : `size` contre `count`.
- Renforcement : drills **D03 Alcohol Consumption**, **D03 Occupation**, **D03 Regiment**. Les versions d'origine de certaines solutions échouent en Pandas 3 (moyenne sur des colonnes texte) : les traductions françaises le signalent.

## À retenir

- `groupby(cle)[colonne].agg(...)` = `SELECT cle, AGG(colonne) ... GROUP BY cle`.
- Les clés deviennent l'index ; `as_index=False` ou `reset_index()` les remettent en colonnes.
- Agrégations nommées pour des résultats lisibles ; `transform` pour une valeur de groupe par ligne.
- `HAVING` = filtrer le résultat de l'agrégation.
- `pivot_table` et `crosstab` pour les tableaux croisés.
- Toujours se demander ce qu'est une ligne avant de compter.